In [1]:
import pandas as pd
import re
def parse_hms_orig(s):
    parts = s.replace("h", "").replace("m", "").replace("s", "").split()
    h, m, sec = map(float, parts)
    return h + m/60 + sec/3600  # decimal hours

def parse_hms(s):
    parts=re.findall(r"([\d]+)[\D]{2}([\d]+)[\D]{2}([\d]+[\.][\d]+)",s)
    if len(parts)==1:
        deg=float(parts[0][0])
        minu=float(parts[0][1])
        sec=float(parts[0][2])
        return (deg + minu/60.0 + sec/3600.0)
    else:
        return 0.0
         
def parse_dec(s):
    parts=re.findall(r"([+-])([\d]+)[\D]{2}([\d]+)[\D]{2}([\d]+)",s) 
    if len(parts)==1:
        if parts[0][0]=="+":
            sign=1
        else:
            sign=-1
        deg=float(parts[0][1])
        minu=float(parts[0][2])
        sec=float(parts[0][3])
        
    return sign * (deg + minu/60.0 + sec/3600.0)

df  =pd.read_csv("col.txt",sep="\t")

df.rename(columns={"Col #":"Id","NGC/Other Cat.":"NGC_Other","m ( v/p)":"mag_v_p","# Stars":"SC","n":"Note"},inplace=True)

df['NGC_Other'] = df['NGC_Other'].fillna("") #There were a few which are not NGC

df['mag_v_p'] = df['mag_v_p'].fillna(10) # There is only 1 Id 240 
df['Size'] = df['Size'].fillna(10) # There is only 1 Id 240 
df['Note'] = df['Note'].fillna(0) # 403 without a Note
# So now we have no Na Values

# extract text inside parentheses
df["Alt_name"] = df["NGC_Other"].str.extract(r"\((.*?)\)")
df["Alt_name"] = df["Alt_name"].fillna("") # 35 have alternative Names 
# remove parentheses part
df["NGC_IC"] = df["NGC_Other"].str.replace(r"\s*\(.*?\)", "", regex=True)
df["Star_Count"]=df["SC"].str.extract(r"\(?(\d+)\)?") 
df['Star_Count'] = df['Star_Count'].fillna(0) 

df["Mag"]=df["mag_v_p"].str.extract(r"\(?(\d+)\)?")
df['Mag'] = df['Mag'].fillna(8) # 8 without magnitude

df["Mag_Type"]=df["mag_v_p"].str.extract(r"([vp])")
df['Mag_Type'] = df['Mag_Type'].fillna("v") # Just 1
df.drop(columns=['SC','mag_v_p'],inplace=True)


# This is from the C++ Code
# KStars 3.8
KSTARS_TYPE_MAP = {
    # Stars
    "STAR": 1,
    "S": 1,
    "TYCHO": 1,

    # Solar system
    "PLANET": 2,
    "ASTEROID": 10,
    "COMET": 9,
    "SATELLITE": 16,

    # Clusters
    "OPEN": 3,
    "OPEN_CLUSTER": 3,
    "OC": 3,
    "GLOBULAR": 4,
    "GC": 4,
    "GLOBULAR_CLUSTER": 4,

    # Nebulae
    "GASEOUS": 5,
    "HII": 5,
    "DIFFUSE": 5,
    "EMISSION": 5,
    "PLANETARY": 6,
    "PN": 6,
    "SUPERNOVA_REMNANT": 7,
    "SNR": 7,

    # Galaxies
    "GALAXY": 8,
    "G": 8,
    "CLUSTER": 14,  # galaxy clusters
    "QUASAR": 15,
}
 

# Note: Mapping was obtained by 
#    df['Class'].value_counts()
# This is not DYNAMIC 

mapping = {
    "Plei": KSTARS_TYPE_MAP["OPEN_CLUSTER"],
    "Praes": KSTARS_TYPE_MAP["OPEN_CLUSTER"],
    "Neb": KSTARS_TYPE_MAP["EMISSION"],
    "μNorm":  KSTARS_TYPE_MAP["OPEN_CLUSTER"], 
    "Glob": KSTARS_TYPE_MAP["GLOBULAR"],
    "nl": KSTARS_TYPE_MAP["OPEN_CLUSTER"], 
    "Chain": KSTARS_TYPE_MAP["OPEN_CLUSTER"],
    "Neb?": KSTARS_TYPE_MAP["EMISSION"],
    "(Neb)": KSTARS_TYPE_MAP["EMISSION"],
}

df["KStars_Type"] = df["Class"].map(mapping)


In [2]:
df

,Id,NGC_Other,Con,RA,DEC,Size,Class,Note,Alt_name,NGC_IC,Star_Count,Mag,Mag_Type,KStars_Type
0,1,103,Cas,00h 25m 17.4s,+61º 19′ 19″,5,Plei,0.0,,103,30,9,v,3
1,2,129,Cas,00h 29m 54.1s,+60º 12′ 35″,21,Plei,0.0,,129,20,6,v,3
2,3,133,Cas,00h 31m 16.9s,+63º 21′ 10″,7,Plei,0.0,,133,20,9,v,3
3,4,136,Cas,00h 31m 30.7s,+61º 30′ 34″,1.2,Praes,0.0,,136,20,11,p,3
4,5,146,Cas,00h 33m 03.9s,+63º 18′ 33″,6,Plei,0.0,,146,20,9,v,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466,467 (156a),(Mel 72 per A/H),Mon,07h 38m 29.0s,-10º 33′ 00″,5,Praes,15.0,Mel 72 per A/H,,40,10,v,3
467,468 (364a),,Sgr,18h 06m 36.0s,-27º 28′ 00″,0.9,Plei,0.0,,,8,11,v,3
468,469 (370a),,Sgr,18h 16m 33.0s,-18º 18′ 34″,2.6,Plei,0.0,,,51,9,v,3
469,470 (442a),IC5146,Cyg,21h 53m 24.0s,+47º 16′ 00″,9,Neb,0.0,,IC5146,110,7,v,5


In [3]:
df["RA_degrees"] = df["RA"].apply(parse_hms)/24.0*360.0


In [4]:
df["DEC_degrees"] = df["DEC"].apply(parse_dec)
df

,Id,NGC_Other,Con,RA,DEC,Size,Class,Note,Alt_name,NGC_IC,Star_Count,Mag,Mag_Type,KStars_Type,RA_degrees,DEC_degrees
0,1,103,Cas,00h 25m 17.4s,+61º 19′ 19″,5,Plei,0.0,,103,30,9,v,3,6.322500,61.321944
1,2,129,Cas,00h 29m 54.1s,+60º 12′ 35″,21,Plei,0.0,,129,20,6,v,3,7.475417,60.209722
2,3,133,Cas,00h 31m 16.9s,+63º 21′ 10″,7,Plei,0.0,,133,20,9,v,3,7.820417,63.352778
3,4,136,Cas,00h 31m 30.7s,+61º 30′ 34″,1.2,Praes,0.0,,136,20,11,p,3,7.877917,61.509444
4,5,146,Cas,00h 33m 03.9s,+63º 18′ 33″,6,Plei,0.0,,146,20,9,v,3,8.266250,63.309167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466,467 (156a),(Mel 72 per A/H),Mon,07h 38m 29.0s,-10º 33′ 00″,5,Praes,15.0,Mel 72 per A/H,,40,10,v,3,114.620833,-10.550000
467,468 (364a),,Sgr,18h 06m 36.0s,-27º 28′ 00″,0.9,Plei,0.0,,,8,11,v,3,271.650000,-27.466667
468,469 (370a),,Sgr,18h 16m 33.0s,-18º 18′ 34″,2.6,Plei,0.0,,,51,9,v,3,274.137500,-18.309444
469,470 (442a),IC5146,Cyg,21h 53m 24.0s,+47º 16′ 00″,9,Neb,0.0,,IC5146,110,7,v,5,328.350000,47.266667


In [5]:
ex_dec="+61º 19′ 19″"
ex_ra="22h 07m 06.0s"

import re
re.findall(r"([+-])([\d]+)[\D]{2}([\d]+)[\D]{2}([\d]+)",ex_dec)  # This is the Dec format i.e. ex_dec

re.findall(r"([\d]+)[\D]{2}([\d]+)[\D]{2}([\d]+[\.][\d]+)",ex_ra) # This is the HA Format i.e. ex_ha

[('22', '07', '06.0')]